# Faruq-v3 — AF2 vs CLAHE classical-enhancement control

Frozen validation-only control: `D0FT vs CLAHE_LAB vs AF2`, seeds 42/123/2026. CLAHE is fixed to RGB→LAB, L-channel only, clipLimit=3.0, tileGridSize=8×8. Test must remain unavailable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-clahe-control'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for key in list(sys.modules):
    if key == 'coffee_detector' or key.startswith('coffee_detector.'):
        sys.modules.pop(key, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


In [ ]:
import cv2, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
AF2_CONFIRMATION = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
D0_42 = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
D0_123 = require_project_artifact(PROJECT_ROOT, REQUIRED[3])
D0_2026 = require_project_artifact(PROJECT_ROOT, REQUIRED[4])
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-af2-vs-clahe-control-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU    :', torch.cuda.get_device_name(0))
print('OpenCV :', cv2.__version__)
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', OUTPUT_ROOT)
for seed in (42, 123, 2026):
    run_dir = OUTPUT_ROOT / 'CLAHE_LAB' / f'CLAHE_LAB_seed{seed}'
    csv_path = run_dir / 'results.csv'
    epochs = 0
    if csv_path.is_file():
        import pandas as pd
        epochs = len(pd.read_csv(csv_path))
    print(f'CLAHE seed {seed}: {epochs}/50')


## Jalankan frozen control

Tiga CLAHE run dilatih berurutan dari checkpoint D0 seed-matched. AF2 dan D0FT tidak dilatih ulang; metrik frozen tiga-seed dibaca dari confirmation result yang sudah PASS. Runner dapat resume dari `last.pt`.


In [ ]:
import csv

command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_af2_clahe_control',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--af2-confirmation', str(AF2_CONFIRMATION),
    '--d0-checkpoints', str(D0_42), str(D0_123), str(D0_2026),
    '--output-root', str(OUTPUT_ROOT),
    '--seeds', '42', '123', '2026',
    '--device', '0', '--authorize-training',
]
RUN_LOG = Path('/content/af2_vs_clahe_control.log')
print('MENJALANKAN AF2-vs-CLAHE frozen control.', flush=True)
with RUN_LOG.open('a', encoding='utf-8', buffering=1) as log_stream:
    process = subprocess.Popen(command, cwd=REPO, text=True, stdout=log_stream, stderr=subprocess.STDOUT)
    while process.poll() is None:
        statuses = []
        for seed in (42, 123, 2026):
            csv_path = OUTPUT_ROOT / 'CLAHE_LAB' / f'CLAHE_LAB_seed{seed}/results.csv'
            epochs = 0
            if csv_path.is_file():
                try:
                    with csv_path.open(newline='', encoding='utf-8') as stream:
                        epochs = sum(1 for _ in csv.DictReader(stream))
                except Exception:
                    epochs = 0
            statuses.append(f'CLAHE-{seed}={epochs}/50')
        print('[STATUS]', ', '.join(statuses), flush=True)
        time.sleep(60)
    return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(RUN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-160:])
    print(tail)
    raise RuntimeError(f'AF2-vs-CLAHE control gagal, return code={return_code}; log={RUN_LOG}')
print('SELESAI. Log:', RUN_LOG)


In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/af2_vs_clahe_classical_enhancement_control.json'
assert SUMMARY.is_file(), f'Control belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = []
for seed, values in result['per_seed'].items():
    for model in ('D0FT', 'CLAHE_LAB', 'AF2'):
        rows.append({'seed': seed, 'model': model, **values[model]})
display(pd.DataFrame(rows).style.format({
    'macro_map50_95': '{:.2%}',
    'bottom3_class_map50_95': '{:.2%}',
    'worst_class_map50_95': '{:.2%}',
}))
print('\nDECISIONS:')
print(json.dumps(result['decisions'], indent=2))
print('INTERPRETATION:', result['interpretation'])
print('SUMMARY       :', SUMMARY)
print('Kirim tabel ini. Jangan buka test.')
